# AI-Powered STAAD.Pro .std Generator
## Full 3D PEB Models with Unity Ratio & Serviceability Checks

This notebook demonstrates an **AI-enhanced, code-driven pipeline** that reads SIJCON-style QRF JSON files and produces **production-ready STAAD.Pro `.std` command files** with:

- Full 3D portal-frame geometry (columns, rafters, purlins, **multi-row girts**, bracing, endwall columns, **mezzanine floors**, **canopy**, **framed openings**, **jack beams**, **floor joists**, **cage ladder**)
- Dead / Live / Wind (±Z ±X) / Seismic (±X) / Crane / **Mezzanine Dead+Live** — up to 10 load cases
- **17+ LRFD load combinations** per ASCE 7 / IS 875 (including longitudinal wind + reverse seismic)
- Steel design CHECK CODE with **UR 0.9–1.0 targeting** (RATIO 0.95)
- **AI/ML Innovation**: LLM-assisted QRF parsing + load-aware section optimizer + PyNite FEA verification
- Serviceability deflection checks with parsed DFF limits
- Auto-detected design code (AISC UNIFIED 2010 / IS800 LSD)
- **BOQ with system-grouped takeoff + 10% fabrication allowance + regional costing**

## 1. Setup & Imports

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

# Install the pipeline package from GitHub + plotly
try:
    from staad_generator._version import __version__
    print(f"staad_generator already installed (v{__version__})")
except ImportError:
    print("Installing staad_generator from GitHub...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "git+https://github.com/ladyFaye1998/staad-pro-3d-generator.git", "-q"
    ])
subprocess.check_call([sys.executable, "-m", "pip", "install", "plotly", "huggingface_hub", "-q"])

# Locate data: Kaggle dataset > local > clone from GitHub
KAGGLE_DS = Path("/kaggle/input/staad-pro-3d-generator")
DATA_DIR = None
for candidate in [KAGGLE_DS / "data", KAGGLE_DS, Path("data")]:
    if candidate.exists() and list(candidate.glob("*.json")):
        DATA_DIR = candidate
        break

if DATA_DIR is None:
    print("Cloning data from GitHub...")
    subprocess.check_call(["git", "clone", "--depth", "1",
        "https://github.com/ladyFaye1998/staad-pro-3d-generator.git", "_repo"])
    DATA_DIR = Path("_repo/data")

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Data directory : {DATA_DIR.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"JSON files found: {len(list(DATA_DIR.glob('*.json')))}")

In [ ]:
# Import the pipeline modules
from staad_generator.spec import spec_from_json_path, format_spec_summary
from staad_generator.geometry import build_frame
from staad_generator.writer import build_std_text
from staad_generator.validate import validate_frame_or_raise
from staad_generator.boq import estimate_boq, format_boq
from staad_generator._version import __version__

print(f"staad_generator v{__version__}")

## 2. Pipeline Overview

```
QRF JSON ──► spec_from_json_path() ──► BuildingSpec
                                            │
                                            ▼
                                     build_frame()  ──► FrameModel (joints + members)
                                            │
                                     validate_frame_or_raise()
                                            │
                                            ▼
                                     build_std_text() ──► STAAD .std text
                                            │
                                     estimate_boq()  ──► BOQ summary
                                            │
                                            ▼
                                      Write .std file
```

## 3. Convert All Competition Files

In [ ]:
results = []

for jp in sorted(DATA_DIR.glob("*.json")):
    print(f"\n{'='*70}")
    print(f"Processing: {jp.name}")
    print(f"{'='*70}")
    
    # Step 1: Parse QRF → BuildingSpec
    spec = spec_from_json_path(jp)
    
    # Step 2: Build 3D frame geometry
    fm = build_frame(spec)
    
    # Step 3: Validate
    validate_frame_or_raise(fm)
    
    # Step 4: Print spec summary
    print(format_spec_summary(spec, n_joints=len(fm.joint_coords), n_members=len(fm.members)))
    
    # Step 5: BOQ estimate
    boq = estimate_boq(spec, fm)
    print(f"\n{format_boq(boq)}")
    
    # Step 6: Generate .std text
    std_text = build_std_text(spec, fm)
    
    # Step 7: Write output
    out_path = OUTPUT_DIR / f"{jp.stem}.std"
    out_path.write_text(std_text, encoding="utf-8", newline="\n")
    
    n_lines = std_text.count("\n")
    print(f"\n  → Written: {out_path}  ({n_lines} lines, {len(std_text)} bytes)")
    
    results.append({
        "file": jp.stem,
        "joints": len(fm.joint_coords),
        "members": len(fm.members),
        "tonnes": boq.total_tonnes,
        "design_code": spec.design_code,
        "lines": n_lines,
    })

print(f"\n\n{'='*70}")
print(f"Done: {len(results)} files converted.")

## 4. Results Summary

In [ ]:
print(f"{'File':<35} {'Joints':>6} {'Members':>7} {'Tonnes':>7} {'Code':<20} {'Lines':>5}")
print("-" * 85)
for r in results:
    print(f"{r['file']:<35} {r['joints']:>6} {r['members']:>7} {r['tonnes']:>7.2f} {r['design_code']:<20} {r['lines']:>5}")
print("-" * 85)
total_t = sum(r['tonnes'] for r in results)
print(f"{'TOTAL':<35} {'':>6} {'':>7} {total_t:>7.2f}")

## 5. Detailed Example: Single File Walkthrough

Let's walk through one file in detail to show every stage of the pipeline.

In [ ]:
# Pick the first competition file (not example_minimal)
demo_files = [f for f in sorted(DATA_DIR.glob("*.json")) if "minimal" not in f.stem]
demo_path = demo_files[0] if demo_files else list(DATA_DIR.glob("*.json"))[0]
print(f"Demo file: {demo_path.name}\n")

# Show raw JSON structure
with open(demo_path, encoding="utf-8") as f:
    raw = json.load(f)

# Show top-level keys
def show_structure(d, indent=0):
    if isinstance(d, dict):
        for k in list(d.keys())[:8]:
            v = d[k]
            if isinstance(v, dict):
                print(f"{'  '*indent}{k}: {{...}} ({len(v)} keys)")
            elif isinstance(v, list):
                print(f"{'  '*indent}{k}: [...] ({len(v)} items)")
            else:
                print(f"{'  '*indent}{k}: {str(v)[:80]}")

show_structure(raw)

In [ ]:
# Parse QRF and show extracted spec
spec = spec_from_json_path(demo_path)
fm = build_frame(spec)
validate_frame_or_raise(fm)

print("=== Extracted Building Specification ===")
print(f"  Name:          {spec.name}")
print(f"  Span (width):  {spec.span_width_m:.2f} m")
print(f"  Length:         {spec.n_bays * spec.bay_length_m:.2f} m  ({spec.n_bays} bays)")
print(f"  Eave height:   {spec.eave_height_m:.2f} m")
print(f"  Roof slope:    1:{1/spec.roof_slope_ratio:.1f}  ({spec.roof_slope_ratio:.4f})")
print(f"  Design code:   {spec.design_code}")
print(f"  Fy:            {spec.fyld_mpa:.0f} MPa")
print(f"  Seismic Ah:    {spec.seismic_ah}")
print(f"  Wind pressure: {spec.wind_pressure_kn_m2:.3f} kN/m²")
print(f"  Crane load:    {spec.crane_load_kn:.1f} kN")
print(f"  Deflection:    V={spec.defl_frame_vertical:.0f}  H={spec.defl_frame_lateral:.0f}  P={spec.defl_purlin:.0f}")
print(f"\n  3D Model: {len(fm.joint_coords)} joints, {len(fm.members)} members")
if spec.bay_spacings:
    print(f"  Non-uniform bays: {spec.bay_spacings}")

## 5.5 Interactive 3D Wireframe

Visualize the generated frame model with color-coded members.

In [ ]:
import plotly.graph_objects as go

KIND_COLORS = {
    "column": "#2563eb", "rafter": "#dc2626", "eave_long": "#f59e0b",
    "ridge_long": "#f59e0b", "purlin": "#10b981", "girt": "#8b5cf6",
    "roof_brace": "#f97316", "wall_brace": "#f97316", "endwall_col": "#06b6d4",
}
KIND_LABELS = {
    "column": "Columns", "rafter": "Rafters", "eave_long": "Eave Beams",
    "ridge_long": "Ridge Beams", "purlin": "Purlins", "girt": "Wall Girts",
    "roof_brace": "Roof Braces", "wall_brace": "Wall Braces", "endwall_col": "Endwall Cols",
}

traces = {}
for mid, n1, n2, kind in fm.members:
    if n1 not in fm.joint_coords or n2 not in fm.joint_coords:
        continue
    x1, y1, z1 = fm.joint_coords[n1]
    x2, y2, z2 = fm.joint_coords[n2]
    if kind not in traces:
        traces[kind] = {"x": [], "y": [], "z": []}
    t = traces[kind]
    t["x"].extend([x1, x2, None])
    t["y"].extend([z1, z2, None])
    t["z"].extend([y1, y2, None])

fig3d = go.Figure()
for kind, coords in traces.items():
    fig3d.add_trace(go.Scatter3d(
        x=coords["x"], y=coords["y"], z=coords["z"],
        mode="lines", name=KIND_LABELS.get(kind, kind),
        line=dict(color=KIND_COLORS.get(kind, "#888"), width=4 if kind in ("column", "rafter") else 2),
    ))

fig3d.update_layout(
    scene=dict(xaxis_title="Length (m)", yaxis_title="Width (m)", zaxis_title="Height (m)",
               aspectmode="data", camera=dict(eye=dict(x=1.5, y=1.5, z=0.8))),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
    margin=dict(l=0, r=0, t=30, b=0), height=550, template="plotly_dark",
    title=f"3D Frame: {spec.name} ({len(fm.joint_coords)} joints, {len(fm.members)} members)",
)
fig3d.show()

In [ ]:
boq = estimate_boq(spec, fm)
kinds = list(boq.by_kind.keys())
values = [boq.by_kind[k] / 1000.0 for k in kinds]
labels = [KIND_LABELS.get(k, k) for k in kinds]
colors = [KIND_COLORS.get(k, "#888") for k in kinds]

fig_boq = go.Figure(go.Bar(
    x=values, y=labels, orientation="h", marker_color=colors,
    text=[f"{v:.1f} t" for v in values], textposition="auto",
))
fig_boq.update_layout(
    title=f"Steel BOQ: {boq.total_tonnes:.2f} tonnes ({boq.member_count} members)",
    xaxis_title="Tonnes", yaxis=dict(autorange="reversed"),
    height=350, template="plotly_dark",
)
fig_boq.show()

In [ ]:
# Generate and show the .std output (first 40 + last 50 lines)
std_text = build_std_text(spec, fm)
lines = std_text.splitlines()
print(f"Total .std lines: {len(lines)}\n")

print("--- HEADER + GEOMETRY (first 40 lines) ---")
for line in lines[:40]:
    print(line)

print(f"\n... ({len(lines) - 90} lines omitted) ...\n")

print("--- DESIGN + SERVICEABILITY (last 50 lines) ---")
for line in lines[-50:]:
    print(line)

In [ ]:
# BOQ estimate
boq = estimate_boq(spec, fm)
print(format_boq(boq))

## 6. Validation: .std File Integrity Checks

We verify that every generated `.std` file contains all required STAAD.Pro blocks.

In [ ]:
REQUIRED_BLOCKS = [
    "STAAD SPACE",
    "JOINT COORDINATES",
    "MEMBER INCIDENCES",
    "MEMBER PROPERTY",
    "DEFINE MATERIAL START",
    "ISOTROPIC STEEL",
    "END DEFINE MATERIAL",
    "CONSTANTS",
    "SUPPORTS",
    "MEMBER TRUSS",
    "MEMBER RELEASE",
    "LOAD 1",
    "LOAD 2",
    "LOAD COMB",
    "PERFORM ANALYSIS",
    "PRINT SUPPORT REACTION",
    "CHECK CODE",
    "SELECT",
    "RATIO 0.95",
    "LOAD LIST",
    "DFF",
    "START GROUP DEFINITION",
    "END GROUP DEFINITION",
    "WIND +X (LONGITUDINAL)",
    "FINISH",
]

all_ok = True
for std_path in sorted(OUTPUT_DIR.glob("*.std")):
    text = std_path.read_text(encoding="utf-8")
    missing = [b for b in REQUIRED_BLOCKS if b not in text]
    status = "PASS" if not missing else f"FAIL (missing: {missing})"
    print(f"  {std_path.name:<40} {status}")
    if missing:
        all_ok = False

print(f"\nOverall: {'ALL PASSED' if all_ok else 'SOME FAILED'}")

## 7. Key Innovations

1. **Robust QRF parsing** — Handles SIJCON bracket notation (`[5@8.700 m]`), mm dimensions, c/c preferences, and fuzzy row matching for inconsistent field names.

2. **Multi-directional loading** — Four wind cases (±Z transverse, ±X longitudinal) + two seismic directions (±X) producing 17+ LRFD combinations.

3. **Auto-detected design code** — Picks `AISC UNIFIED 2010` vs `IS800 LSD` based on QRF content.

4. **Full 4-stage design workflow** — CHECK CODE → SELECT (RATIO 0.95) → SLS deflection check → final CHECK CODE.

5. **Proper STAAD.Pro syntax** — `STAAD SPACE` first, `START GROUP DEFINITION` / `MEMBER` / `END GROUP DEFINITION`, inline `MEMBER TRUSS`, correct `TAPERED` properties.

6. **Enhanced BOQ** — System-grouped takeoff (Primary, Secondary, Bracing, Mezzanine) + 10% fabrication/connections allowance + regional costing.

7. **Multiple girt rows** — Wall girt count adapts to building height (up to 4 rows for tall buildings).

8. **Mezzanine joint merging** — Base joints shared with portal frame columns for structural continuity.

In [ ]:
print("\nAll output files:")
for p in sorted(OUTPUT_DIR.glob("*.std")):
    size = p.stat().st_size
    print(f"  {p.name:<40} {size:>8,} bytes")